In [1]:
import kairo
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
from pathlib import Path
import os

load_dotenv(Path("..") / ".env")

True

In [2]:
log = kairo.read_log("../data/drifted_logs/time_fast/sudden/bank_account_closure/bank_account_closure_time_fast_x05_sudden.xes",
    case_id = "case:concept:name",
    activity = "concept:name",
    timestamp = "time:timestamp",
    #event_id = "event:uidd",
    start_timestamp = "start:timestamp",
    resource = "org:resource",
    #event_duration = "event:runtime_min")
    )
log.head()

,time:timestamp,org:resource,lifecycle:transition,concept:name,CLOSURE_TYPE,start_timestamp,CLOSURE_REASON,case:concept:name,org:role
0,2017-05-29 08:49:25+00:00,6,complete,Request created,Client Recess,2017-05-29 08:47:46+00:00,1 - Client lost,case_000000,APPLICANT
1,2017-05-29 08:52:49+00:00,7,complete,Evaluating Request (NO registered letter),Client Recess,2017-05-29 08:49:25+00:00,1 - Client lost,case_000000,DIRECTOR
2,2017-05-29 09:02:39+00:00,7,complete,Service closure Request with network responsib...,Client Recess,2017-05-29 08:56:04+00:00,1 - Client lost,case_000000,APPLICANT
3,2017-05-29 11:23:33+00:00,BOF,complete,Service closure Request with BO responsibility,Client Recess,2017-05-29 09:02:39+00:00,1 - Client lost,case_000000,BACK-OFFICE
4,2017-05-31 09:26:08+00:00,7,complete,Network Adjustment Requested,Client Recess,2017-05-29 11:23:33+00:00,1 - Client lost,case_000000,APPLICANT


In [3]:
stats = kairo.compute_log_stats(log)
stats

{'cases': 32429,
 'events': 212721,
 'activities': 15,
 'resources': 654,
 'start': Timestamp('2017-05-29 08:49:25+0000', tz='UTC'),
 'end': Timestamp('2019-03-06 16:38:59+0000', tz='UTC'),
 'span_minutes': 930709.5666666667,
 'tpt_days_min': 0.0,
 'tpt_days_mean': 13.45629664160194,
 'tpt_days_median': 6.074346064814815,
 'tpt_days_max': 503.5870949074074,
 'length_min': 2,
 'length_mean': 6.559591723457399,
 'length_median': 7.0,
 'length_max': 23,
 'activity_counts': concept:name
 Request created                                        32377
 Service closure Request with BO responsibility         30130
 Pending Request for Reservation Closure                29151
 Request completed with account closure                 29031
 Pending Liquidation Request                            29000
 Service closure Request with network responsibility    25009
 Authorization Requested                                12675
 Evaluating Request (NO registered letter)              11080
 Back-Office Adj

In [4]:
activity_plot = kairo.plot_activity_counts(stats)
activity_plot.show()

In [5]:
# one row per calendar window, the case attributes with their kind,
# e.g. {"case:amount": "numerical", "case:region": "categorical"}
features = kairo.compute_features_inter(log, window="1D",
    case_attributes={"CLOSURE_TYPE": "categorical"},
    stall_threshold="1D")
features

,time:timestamp,active_cases,new_arrivals,completions,events_per_case,mean_delta_t,std_delta_t,stalled_cases,attr_shares:CLOSURE_TYPE=Bank Recess,attr_shares:CLOSURE_TYPE=Client Recess,attr_shares:CLOSURE_TYPE=Inheritance,attr_shares:CLOSURE_TYPE=NAN,attr_shares:CLOSURE_TYPE=Porting
0,2017-05-29 00:00:00+00:00,1.0,1.0,0.0,4.000000,51.377778,77.595220,0.0,0.000000,1.000000,0.000000,0.0,0.0
1,2017-05-30 00:00:00+00:00,0.0,0.0,0.0,0.000000,0.000000,0.000000,1.0,0.000000,0.000000,0.000000,0.0,0.0
2,2017-05-31 00:00:00+00:00,1.0,0.0,0.0,1.000000,2762.583333,0.000000,0.0,0.000000,1.000000,0.000000,0.0,0.0
3,2017-06-01 00:00:00+00:00,0.0,0.0,0.0,0.000000,0.000000,0.000000,1.0,0.000000,0.000000,0.000000,0.0,0.0
4,2017-06-02 00:00:00+00:00,0.0,0.0,0.0,0.000000,0.000000,0.000000,1.0,0.000000,0.000000,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
642,2019-03-02 00:00:00+00:00,42.0,0.0,35.0,1.833333,1359.383333,1447.959514,12.0,0.324675,0.376623,0.298701,0.0,0.0
643,2019-03-03 00:00:00+00:00,5.0,0.0,5.0,2.000000,1659.955000,1749.777453,14.0,0.800000,0.200000,0.000000,0.0,0.0
644,2019-03-04 00:00:00+00:00,45.0,32.0,45.0,2.022222,778.321186,1492.725463,1.0,0.373626,0.417582,0.208791,0.0,0.0
645,2019-03-05 00:00:00+00:00,18.0,17.0,18.0,2.000000,247.431579,1078.529248,0.0,0.500000,0.388889,0.111111,0.0,0.0


In [6]:
std_features = kairo.standardize(features, method="minmax")
std_features

,time:timestamp,active_cases,new_arrivals,completions,events_per_case,mean_delta_t,std_delta_t,stalled_cases,attr_shares:CLOSURE_TYPE=Bank Recess,attr_shares:CLOSURE_TYPE=Client Recess,attr_shares:CLOSURE_TYPE=Inheritance,attr_shares:CLOSURE_TYPE=NAN,attr_shares:CLOSURE_TYPE=Porting
0,2017-05-29 00:00:00+00:00,0.000720,0.002632,0.000000,0.800000,0.000305,0.000427,0.000000,0.000000,1.000000,0.000000,0.0,0.0
1,2017-05-30 00:00:00+00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000326,0.000000,0.000000,0.000000,0.0,0.0
2,2017-05-31 00:00:00+00:00,0.000720,0.000000,0.000000,0.200000,0.016384,0.000000,0.000000,0.000000,1.000000,0.000000,0.0,0.0
3,2017-06-01 00:00:00+00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000326,0.000000,0.000000,0.000000,0.0,0.0
4,2017-06-02 00:00:00+00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000326,0.000000,0.000000,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
642,2019-03-02 00:00:00+00:00,0.030259,0.000000,0.048343,0.366667,0.008062,0.007977,0.003906,0.324675,0.376623,0.298701,0.0,0.0
643,2019-03-03 00:00:00+00:00,0.003602,0.000000,0.006906,0.400000,0.009844,0.009640,0.004557,0.800000,0.200000,0.000000,0.0,0.0
644,2019-03-04 00:00:00+00:00,0.032421,0.084211,0.062155,0.404444,0.004616,0.008223,0.000326,0.373626,0.417582,0.208791,0.0,0.0
645,2019-03-05 00:00:00+00:00,0.012968,0.044737,0.024862,0.400000,0.001467,0.005942,0.000000,0.500000,0.388889,0.111111,0.0,0.0


In [7]:
pca = kairo.compute_pca(std_features, num_components=None)
pca_plot = kairo.plot_pca_variances(pca)
pca_plot.show()
#pca

In [8]:
reconstruction_errors = kairo.compute_pca_reconstruction_errors(std_features, pca)
reconstruction_plot = kairo.plot_pca_reconstruction_errors(reconstruction_errors)
reconstruction_plot.show()

In [9]:
compressed_features = kairo.apply_pca(std_features, pca, cut_component=5)
compressed_features

,time:timestamp,pc_1,pc_2,pc_3,pc_4,pc_5
0,2017-05-29 00:00:00+00:00,0.365454,0.681563,-0.145436,0.087526,-0.074754
1,2017-05-30 00:00:00+00:00,-0.497923,-0.152913,-0.122709,0.003773,-0.085196
2,2017-05-31 00:00:00+00:00,0.174044,0.580329,-0.303763,-0.053445,-0.145675
3,2017-06-01 00:00:00+00:00,-0.497923,-0.152913,-0.122709,0.003773,-0.085196
4,2017-06-02 00:00:00+00:00,-0.497923,-0.152913,-0.122709,0.003773,-0.085196
...,...,...,...,...,...,...
642,2019-03-02 00:00:00+00:00,-0.085889,0.188750,0.196666,0.254081,0.042242
643,2019-03-03 00:00:00+00:00,-0.138299,0.019858,0.019910,0.708840,0.316718
644,2019-03-04 00:00:00+00:00,-0.002418,0.167787,0.124583,0.322425,0.035335
645,2019-03-05 00:00:00+00:00,-0.037966,0.157033,0.049396,0.437452,0.128296


In [10]:
pca_plot_cut = kairo.plot_pca_variances(pca, cut_component=5)
pca_plot_cut.show()

In [ ]:
reconstruction_plot_cut = kairo.plot_pca_reconstruction_errors(reconstruction_errors, cut_component=5)
reconstruction_plot_cut.show()

In [ ]:
# try min_samples around D + 1, D being the number of PCA components, and eps at the knee
k_distances_plot = kairo.plot_dbscan_k_distance(compressed_features, min_samples=6, distance="euclidean")
k_distances_plot.show()

In [ ]:
dbscan = kairo.compute_dbscan(compressed_features, min_samples=6, eps=0.2, distance="euclidean")
clusters = kairo.get_dbscan_clusters(compressed_features, dbscan)
clusters

In [ ]:
clustering_scores = [
    {"type": "DBCV", "value": kairo.compute_dbcv(compressed_features, clusters, distance=dbscan.metric)},
    {"type": "Silhouette", "value": kairo.compute_silhouette(compressed_features, clusters, distance=dbscan.metric)},
    {"type": "Calinski-Harabasz", "value": kairo.compute_calinski_harabasz(compressed_features, clusters)},
]
clustering_scores

In [ ]:
features_and_clusters = pd.concat([compressed_features, clusters], axis=1)
features_and_clusters

In [ ]:
state_distances = kairo.get_dbscan_state_distances(dbscan)
state_distances

In [ ]:
state_freqs = kairo.get_dbscan_state_frequencies(features_and_clusters)
state_freqs

In [ ]:
distances_plot = kairo.plot_dbscan_distances(state_distances)
distances_plot.show()

In [ ]:
color_mapping = kairo.compute_dbscan_color_mapping(dbscan)

In [ ]:
freqs_plot = kairo.plot_dbscan_frequencies(state_freqs, color_mapping)
freqs_plot.show()

In [ ]:
log_trajectory = kairo.get_dbscan_log_trajectory(features_and_clusters,
                                 #start_date="2017-06-01",
                                 #end_date="2018-11-26"
                                 )
log_trajectory

In [ ]:
log_trajectory_plot = kairo.plot_dbscan_log_trajectory(log_trajectory, color_mapping)
log_trajectory_plot

In [ ]:
# the states of the daily windows counted per 30 days
freq_distributions = kairo.compute_state_distributions(features_and_clusters, window="30D")
freq_distributions

In [ ]:
distributions_plot = kairo.plot_state_distributions(freq_distributions, color_mapping)
distributions_plot.show()

In [ ]:
divergences = kairo.compute_divergences(freq_distributions)
divergences

In [ ]:
divergences_plot = kairo.plot_divergences(divergences)
divergences_plot.show()

In [ ]:
window_distances = kairo.compute_window_distances(compressed_features, distance="euclidean", reference="previous")
window_distances

In [ ]:
window_distances_plot = kairo.plot_window_distances(window_distances, distance="euclidean", reference="previous")
window_distances_plot.show()

In [ ]:
abs_log_stats = kairo.abstract_log_stats(stats)
abs_log_stats

In [ ]:
abs_features = kairo.abstract_features(features)
abs_features

In [ ]:
abs_pca = kairo.abstract_pca(pca, cut_component=5)
abs_pca

In [ ]:
abs_states = kairo.abstract_states(state_freqs, state_distances, color_mapping, clustering_scores=clustering_scores)
abs_states

In [ ]:
abs_log_trajectory = kairo.abstract_log_trajectory(log_trajectory)
abs_log_trajectory

In [ ]:
abs_distributions = kairo.abstract_distributions(freq_distributions)
abs_distributions

In [ ]:
abs_divergences = kairo.abstract_divergences(divergences, divergence="kl", reference="previous")
abs_divergences

In [ ]:
abs_window_distances = kairo.abstract_window_distances(window_distances, distance="euclidean", reference="previous")
abs_window_distances

In [ ]:
llm = kairo.LLMConnector(
    provider="custom",
    #api_key=os.environ["ANTHROPIC_API_KEY"],
    model="Qwen/Qwen3-VL-32B-Instruct",
    base_url="http://137.226.117.70:8000/v1"
)

In [ ]:
chat_history = []
total_out_tokens = 0

In [ ]:
# For initial request
messages = kairo.create_messages(provider="custom",
                                system_prompt=kairo.DEFAULT_SYSTEM_PROMPT,
                                prompts=[abs_log_stats, abs_features, abs_pca, abs_states, abs_log_trajectory,
                                         abs_distributions, abs_divergences, abs_window_distances,
                                         "Analyze and summarize the given information. Do not use markdown format, just raw text"],
                                plots=[activity_plot, pca_plot_cut, distances_plot, freqs_plot,
                                       log_trajectory_plot, distributions_plot, divergences_plot, window_distances_plot])
chat_history += messages

num_tokens = kairo.count_input_tokens(provider="custom", messages=chat_history)
num_tokens

In [ ]:
# For follow up request
messages = kairo.create_messages(provider="custom",
                                system_prompt=None,
                                prompts=["what was my previous question very briefly"],
                                plots=[])
chat_history += messages

num_tokens = kairo.count_input_tokens(provider="custom", messages=chat_history)
num_tokens

In [ ]:
response = llm.call(messages=chat_history, max_tokens=64000)

answer = kairo.get_response_text(response)

chat_history.append({"role": "assistant", 
                     "content": answer})

print(answer)

In [ ]:
out_tokens = kairo.count_output_tokens("custom", response)
total_out_tokens += out_tokens
total_out_tokens

In [ ]:
print(kairo.get_chat_history(chat_history))